# 1.1 Data cleaning

Reads the frozen `legal_paragraphs.parquet` and writes `legal_paragraphs_clean.parquet`.
The source file is never modified.

Motivation: the per-document paragraph count is **not iid**. It is wildly overdispersed
(var/mean = 54 where Poisson needs 1) and clustered by issuing court (ICC ~= 0.15).
Part of the extreme tail is genuine variation between courts and case types, but part is
parsing failure. This notebook separates the two: it drops only unambiguous failures and
*flags* everything else so downstream analysis can condition on quality instead of
silently inheriting it.

In [1]:
import hashlib

import numpy as np
import pandas as pd

In [2]:
SOURCE_PATH = "legal_paragraphs.parquet"
CLEAN_PATH = "legal_paragraphs_clean.parquet"
REPORT_PATH = "data_quality_report.csv"

# A well-formed judgment is one run each of introductory -> reasoning -> operative.
EXPECTED_SECTION_RUNS = 3
PARSE_FAILURE_DUP_RATE = 0.30
DEGENERATE_MAX_PARAGRAPHS = 2
EXTREME_LENGTH_MIN_PARAGRAPHS = 300

In [3]:
p_df = pd.read_parquet(SOURCE_PATH)
p_df["document_id"] = p_df["document_id"].astype(str)
p_df = p_df.sort_values(["document_id", "paragraph_index"], kind="stable").reset_index(drop=True)
p_df.shape, p_df.columns.tolist()

((289367, 6),
 ['document_id',
  'paragraph_index',
  'paragraph_order',
  'numbered_text',
  'text',
  'section'])

## Per-document quality profile

In [4]:
by_document = p_df.groupby("document_id", sort=True)

def section_runs(sections):
    return int((sections != sections.shift()).sum())

def document_hash(texts):
    return hashlib.md5("\n".join(texts).encode("utf-8")).hexdigest()

profile = pd.DataFrame({
    "n_paragraphs": by_document.size(),
    "n_section_runs": by_document["section"].apply(section_runs),
    "within_doc_dup_rate": by_document["text"].apply(lambda s: float(s.duplicated().mean())),
    "doc_hash": by_document["text"].apply(document_hash),
})
profile = profile.join(
    pd.crosstab(p_df["document_id"], p_df["section"]).add_prefix("n_")
)
profile.head()

,n_paragraphs,n_section_runs,within_doc_dup_rate,doc_hash,n_introductory,n_operative,n_reasoning
document_id,,,,,,,
116061850,104,3,0.009615,275e88605524492a91110592684717ca,26,25,53
116061851,3,3,0.000000,0f99371bedf5843a74919f80b4ec6165,1,1,1
116061852,2,2,0.000000,ec3ca183ab333c2a9020a1fef4232a89,1,1,0
116061853,3,3,0.000000,4837291c7c8df5289d567f99ffae7582,1,1,1
116061854,3,3,0.000000,23d35bf649fe7cd5a183d935c1493f38,1,1,1


## Flags

`dup_document`, `parse_failure` and `degenerate` are dropped. The rest are kept and carried
downstream as boolean columns: a scrambled section order or an unusually long document is a
property of the judgment, not evidence of a broken parse, and dropping 9% of documents on a
heuristic would bias the sample more than it cleans it.

In [5]:
profile["dup_document"] = profile["doc_hash"].duplicated(keep="first")
profile["parse_failure"] = profile["within_doc_dup_rate"] > PARSE_FAILURE_DUP_RATE
profile["degenerate"] = profile["n_paragraphs"] <= DEGENERATE_MAX_PARAGRAPHS
profile["section_scrambled"] = profile["n_section_runs"] > EXPECTED_SECTION_RUNS
profile["section_missing"] = profile["n_section_runs"] < EXPECTED_SECTION_RUNS
profile["extreme_length"] = profile["n_paragraphs"] >= EXTREME_LENGTH_MIN_PARAGRAPHS

DROP_FLAGS = ["dup_document", "parse_failure", "degenerate"]
KEEP_FLAGS = ["section_scrambled", "section_missing", "extreme_length"]

profile["dropped"] = profile[DROP_FLAGS].any(axis=1)

flag_counts = profile[DROP_FLAGS + KEEP_FLAGS].sum().to_frame("n_documents")
flag_counts["action"] = ["drop"] * len(DROP_FLAGS) + ["flag"] * len(KEEP_FLAGS)
flag_counts

,n_documents,action
dup_document,16,drop
parse_failure,32,drop
degenerate,11,drop
section_scrambled,788,flag
section_missing,681,flag
extreme_length,19,flag


In [6]:
print(f"documents dropped: {int(profile['dropped'].sum())} "
      f"({profile['dropped'].mean():.2%})")
profile.loc[profile["dropped"], ["n_paragraphs", "n_section_runs", "within_doc_dup_rate"] + DROP_FLAGS] \
    .sort_values("n_paragraphs", ascending=False).head(20)

documents dropped: 51 (0.60%)


,n_paragraphs,n_section_runs,within_doc_dup_rate,dup_document,parse_failure,degenerate
document_id,,,,,,
116084033,728,7,0.456044,False,True,False
116102798,727,35,0.515818,False,True,False
116102802,727,34,0.515818,False,True,False
116102801,727,35,0.495186,False,True,False
116067141,346,5,0.343931,False,True,False
116099789,330,3,0.472727,False,True,False
116067139,314,3,0.375796,False,True,False
116085725,307,55,0.530945,False,True,False
116062492,283,3,0.558304,False,True,False


## Write the cleaned dataset

In [7]:
kept = profile.index[~profile["dropped"]]
clean_df = p_df[p_df["document_id"].isin(kept)].copy()

for flag in KEEP_FLAGS:
    clean_df[flag] = clean_df["document_id"].map(profile[flag])
clean_df["n_paragraphs"] = clean_df["document_id"].map(profile["n_paragraphs"]).astype("int32")

# The cleaner only removes whole documents; it never rewrites a paragraph.
expected_rows = int(profile.loc[kept, "n_paragraphs"].sum())
assert len(clean_df) == expected_rows, (len(clean_df), expected_rows)
assert not clean_df.duplicated(["document_id", "paragraph_index"]).any()
merged = clean_df.merge(p_df, on=["document_id", "paragraph_index"], suffixes=("", "_src"))
assert (merged["text"] == merged["text_src"]).all()
assert (merged["section"] == merged["section_src"]).all()
del merged

clean_df.to_parquet(CLEAN_PATH, index=False)
profile.to_csv(REPORT_PATH)
clean_df.shape

(281725, 10)

## Before / after

Cleaning removes the broken documents but **must not** remove the clustering — the
overdispersion and the between-court correlation are properties of how the corpus was
sampled, not artefacts of bad parses. If the numbers below collapsed, the flags would be
throwing away signal.

In [8]:
def count_summary(counts, label):
    counts = np.asarray(counts, dtype=float)
    return pd.Series({
        "n_documents": len(counts),
        "n_paragraphs": counts.sum(),
        "mean": counts.mean(),
        "median": np.median(counts),
        "max": counts.max(),
        "var/mean": counts.var(ddof=1) / counts.mean(),
        "skew": pd.Series(counts).skew(),
    }, name=label)

before = count_summary(profile["n_paragraphs"], "before")
after = count_summary(profile.loc[kept, "n_paragraphs"], "after")
pd.concat([before, after], axis=1).round(2)

,before,after
n_documents,8511.00,8460.00
n_paragraphs,289367.00,281725.00
mean,34.00,33.30
median,28.00,28.00
max,1967.00,1967.00
var/mean,54.21,46.51
skew,24.74,29.49


In [9]:
def icc_by_group(values, groups, min_group_size=10):
    """One-way random-effects ICC of log(values), over groups with enough members."""
    frame = pd.DataFrame({"value": np.log(np.asarray(values, dtype=float)), "group": list(groups)})
    sizes = frame["group"].value_counts()
    frame = frame[frame["group"].isin(sizes[sizes >= min_group_size].index)]
    if frame["group"].nunique() < 2:
        return np.nan, 0, 0

    grouped = frame.groupby("group")["value"]
    k, n = grouped.ngroups, len(frame)
    ss_between = (grouped.size() * (grouped.mean() - frame["value"].mean()) ** 2).sum()
    ss_within = ((frame["value"] - grouped.transform("mean")) ** 2).sum()
    ms_between, ms_within = ss_between / (k - 1), ss_within / (n - k)
    n0 = (n - (grouped.size() ** 2).sum() / n) / (k - 1)
    return (ms_between - ms_within) / (ms_between + (n0 - 1) * ms_within), k, n

In [10]:
# Court is not a column in this dataset, so proxy it with the first paragraph of the
# document, which for most judgments is the court name. 1.3 builds a stricter extractor.
first_paragraph = by_document["text"].first().str.strip().str.upper()

for label, index in [("before", profile.index), ("after", kept)]:
    icc, k, n = icc_by_group(profile.loc[index, "n_paragraphs"], first_paragraph.loc[index])
    print(f"{label:>6}: ICC(log paragraphs | court proxy) = {icc:.3f}  "
          f"over {k} groups / {n} documents")

before: ICC(log paragraphs | court proxy) = 0.195  over 52 groups / 1678 documents
 after: ICC(log paragraphs | court proxy) = 0.173  over 52 groups / 1660 documents


In [11]:
print(profile.loc[kept, KEEP_FLAGS].mean().to_string())
print()
print("paragraph counts by flag (median):")
print(profile.loc[kept].groupby(KEEP_FLAGS)["n_paragraphs"].agg(["size", "median"]).to_string())

section_scrambled    0.090780
section_missing      0.079078
extreme_length       0.001300

paragraph counts by flag (median):
                                                  size  median
section_scrambled section_missing extreme_length              
False             False           False           7023    28.0
                  True            False            669    22.0
True              False           False            757    25.0
                                  True              11   595.0
